In [9]:
"""
PWR Thermal-Hydraulic Core Design — Coupled Iterative Optimizer
NUEN 489 Semester Project
Constraints:  dp ≤ 175 kPa
              T_center ≤ 1000 °C
Objective:    minimize core volume
"""

import math, importlib, warnings
warnings.filterwarnings('ignore')
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.special import j0
from scipy.optimize import minimize, differential_evolution, Bounds
from scipy.interpolate import RegularGridInterpolator
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# fluid property library
@dataclass
class FluidProperties:
    T_C: float; rho: float; mu: float; cp: float; k: float; Pr: float

class WaterPropertyLibrary:
    """15.5 MPa PWR coolant — tables from NIST/Incropera interpolated."""
    def __init__(self):
        table = [
            # T_C   rho    mu        cp      k
            (270, 780.0, 1.15e-4, 5100.0, 0.555),
            (280, 760.0, 1.05e-4, 5150.0, 0.560),
            (290, 745.0, 9.60e-5, 5180.0, 0.565),
            (300, 730.0, 8.90e-5, 5210.0, 0.570),
            (310, 715.0, 8.25e-5, 5250.0, 0.575),
            (320, 700.0, 7.65e-5, 5290.0, 0.580),
            (330, 685.0, 7.10e-5, 5330.0, 0.585),
            (340, 670.0, 6.60e-5, 5380.0, 0.590),
            (350, 655.0, 6.15e-5, 5440.0, 0.595),
        ]
        self.df = pd.DataFrame(table, columns=['T_C','rho','mu','cp','k'])

    def get(self, T_C: float) -> FluidProperties:
        T = float(np.clip(T_C, 270.0, 350.0))
        v = {c: float(np.interp(T, self.df['T_C'], self.df[c])) for c in ['rho','mu','cp','k']}
        Pr = v['cp']*v['mu']/v['k']
        return FluidProperties(T, v['rho'], v['mu'], v['cp'], v['k'], Pr)


# configuration
@dataclass
class PWRConfig:
    #fixed
    power_total: float    = 500e6
    phi_max_cm2_s: float  = 5.0e13
    N_ff_cm3: float       = 8.0e20
    sigma_f_barn: float   = 585.0
    G_f_MeV: float        = 200.0
    pressure_nominal_pa: float = 15.5e6
    delta_p_target_pa: float   = 175e3
    max_fuel_temp_C: float     = 1000.0
    gap_h: float          = 7000.0
    clad_k: float         = 20.0
    clad_thickness: float = 0.65e-3
    Tin_C: float          = 290.0
    target_Tout_C: float  = 325.0
    # design var
    aspect_ratio_H_over_D: float  = 1.00
    rod_pitch_m: float            = 12.6e-3
    fuel_pellet_diameter_m: float = 8.0e-3
    # fixed assembly geometry
    assembly_side_rods: int = 17
    gap_thickness_m: float  = 0.08e-3
    fuel_k: float           = 3.5
    # solver resolution
    nr: int   = 20
    nz: int   = 60
    fuel_nr: int = 80
    max_iter: int = 120
    tol: float   = 1e-6


# core solver
class PWRCoreSolver:
    def __init__(self, cfg: PWRConfig, prop_lib: WaterPropertyLibrary = None):
        self.cfg  = cfg
        self.prop = prop_lib or WaterPropertyLibrary()
        self.results: Dict = {}

    # neutronics
    def _qppp_max(self) -> float:
        c = self.cfg
        return (c.phi_max_cm2_s*1e4) * (c.N_ff_cm3*1e6) * (c.sigma_f_barn*1e-28) * (c.G_f_MeV*1.602176634e-13)

    def _shape_avg(self) -> float:
        r = np.linspace(0,1,500); z = np.linspace(-0.5,0.5,500)
        Rg,Zg = np.meshgrid(r,z,indexing='ij')
        s = j0(2.405*Rg)*np.cos(np.pi*Zg)
        return float(2*np.trapz(np.trapz(s*Rg,z,axis=1),r))

    def neutron_physics(self):
        qm = self._qppp_max();  sa = self._shape_avg()
        V  = self.cfg.power_total/(qm*sa)
        H  = (4*V*self.cfg.aspect_ratio_H_over_D/math.pi)**(1/3)
        D  = H/self.cfg.aspect_ratio_H_over_D;  R = D/2
        nr,nz = self.cfg.nr, self.cfg.nz
        re = np.linspace(0,R,nr+1); ze = np.linspace(-H/2,H/2,nz+1)
        rc = 0.5*(re[:-1]+re[1:]); zc = 0.5*(ze[:-1]+ze[1:])
        Rg,Zg = np.meshgrid(rc,zc,indexing='ij')
        fs   = j0(2.405*Rg/R)*np.cos(np.pi*Zg/H)
        qppp = qm*fs
        cv   = np.zeros_like(qppp); dz = ze[1:]-ze[:-1]
        for i in range(nr):
            cv[i,:] = math.pi*(re[i+1]**2-re[i]**2)*dz
        qppp *= self.cfg.power_total/np.sum(qppp*cv)
        self.results['neutronics'] = dict(
            V=V, H=H, D=D, R=R,
            rc=rc, zc=zc, fs=fs, qppp=qppp, cv=cv)

    # geometry
    def _geometry(self):
        if 'neutronics' not in self.results: self.neutron_physics()
        c = self.cfg
        rf  = c.fuel_pellet_diameter_m/2
        rg  = rf + c.gap_thickness_m
        rcl = rg + c.clad_thickness
        fa  = c.rod_pitch_m**2 - math.pi*rcl**2
        Dh  = 4*fa/(2*math.pi*rcl)
        ap  = c.assembly_side_rods*c.rod_pitch_m
        na  = max(1, int(math.pi*self.results['neutronics']['R']**2/ap**2))
        nr  = c.assembly_side_rods**2 * na
        self.results['geometry'] = dict(
            rf=rf, rg=rg, rcl=rcl, fa=fa, Dh=Dh,
            n_assy=na, n_rods=nr,
            total_flow_area=fa*nr)

    # fluids (Re-Nu-h-T)
    def fluid_dynamics(self):
        if 'geometry' not in self.results: self._geometry()
        neu = self.results['neutronics']
        geo = self.results['geometry']
        c   = self.cfg
        H, nz = neu['H'], c.nz
        ax_pwr = np.sum(neu['qppp']*neu['cv'], axis=0)   # (nz,)
        # guess from target dT
        cp0 = self.prop.get((c.Tin_C+c.target_Tout_C)/2).cp
        m   = c.power_total/(cp0*(c.target_Tout_C-c.Tin_C))
        dp  = 0.0
        hist: List[Dict] = []

        for it in range(1, c.max_iter+1):
            G = m/geo['total_flow_area']
            T  = np.zeros(nz+1); T[0] = c.Tin_C
            rho = np.empty(nz); mu = np.empty(nz)
            cp  = np.empty(nz); kf = np.empty(nz)
            Re  = np.empty(nz); Nu = np.empty(nz); h = np.empty(nz)

            for k in range(nz):
                p = self.prop.get(T[k])
                rho[k]=p.rho; mu[k]=p.mu; cp[k]=p.cp; kf[k]=p.k
                Re[k] = G*geo['Dh']/p.mu                        # Reynolds
                Nu[k] = 0.023*Re[k]**0.8 * p.Pr**0.4            # Dittus-Boelter
                h[k]  = Nu[k]*p.k/geo['Dh']                     # convection coeff
                T[k+1]= T[k] + ax_pwr[k]/(m*p.cp)              # energy balance

            f_ax = 0.3164/Re**0.25
            dz   = H/nz
            dp   = float(np.sum(f_ax*(dz/geo['Dh'])*(G**2/(2*rho))))
            err  = (dp-c.delta_p_target_pa)/c.delta_p_target_pa
            hist.append(dict(iter=it, m=m, G=G, Re_avg=Re.mean(), Nu_avg=Nu.mean(),
                             h_avg=h.mean(), Tout=T[-1], dp_kPa=dp/1e3, err=err))
            if abs(err) < c.tol: break
            m *= math.sqrt(c.delta_p_target_pa/max(dp,1e-9))

        q_lin_z = (c.power_total/(geo['n_rods']*H)) * (ax_pwr/ax_pwr.mean())
        Tc = 0.5*(T[:-1]+T[1:])
        self.results['fluid'] = dict(
            m=m, dp=dp, T_bulk=T, Tc=Tc,
            Re=Re, Nu=Nu, h=h,
            q_lin=q_lin_z, hist=pd.DataFrame(hist))

    # heat transfer radial
    def _radial_solve(self, q_lin, T_cool, h_conv) -> Dict:
        c   = self.cfg
        geo = self.results['geometry']
        rf, rg, rcl = geo['rf'], geo['rg'], geo['rcl']
        n = c.fuel_nr; dr = rf/n
        r = (np.arange(n)+0.5)*dr
        A = sparse.lil_matrix((n,n)); b = np.zeros(n)
        qppp = q_lin/(math.pi*rf**2)
        for i in range(n):
            rw = max(r[i]-0.5*dr,0.0); re2 = r[i]+0.5*dr
            Aw = 2*math.pi*rw;  Ae = 2*math.pi*re2
            aw = c.fuel_k*Aw/(dr if i>0 else dr/2) if i>0 else 0.0
            ae = c.fuel_k*Ae/dr
            vol = math.pi*(re2**2-rw**2)
            A[i,i]  = aw+ae
            if i>0:   A[i,i-1] = -aw
            if i<n-1: A[i,i+1] = -ae
            b[i] = qppp*vol
        R_gap  = 1/(c.gap_h*2*math.pi*rf)
        R_clad = math.log(rcl/rg)/(2*math.pi*c.clad_k)
        R_conv = 1/(h_conv*2*math.pi*rcl)
        A[n-1,n-1] += 1/(R_gap+R_clad+R_conv)
        b[n-1]     += T_cool/(R_gap+R_clad+R_conv)
        Tf = spsolve(A.tocsr(), b)
        return dict(r=r, Tf=Tf, Tc=float(Tf[0]), Ts=float(Tf[-1]),
                    Tg=float(Tf[-1]-q_lin*R_gap),
                    Tco=float(Tf[-1]-q_lin*R_gap-q_lin*R_clad))

    def heat_transfer(self):
        if 'fluid' not in self.results: self.fluid_dynamics()
        fl  = self.results['fluid']
        geo = self.results['geometry']
        pk  = int(np.argmax(fl['q_lin']))
        q, Tc_pk, h_pk = float(fl['q_lin'][pk]), float(fl['Tc'][pk]), float(fl['h'][pk])
        hist = []
        for it in range(1, self.cfg.max_iter+1):
            rad  = self._radial_solve(q, Tc_pk, h_pk)
            hist.append(dict(iter=it, Re_pk=float(fl['Re'][pk]),
                             Nu_pk=float(fl['Nu'][pk]),
                             h_pk=h_pk, Tcool=Tc_pk, Tcen=rad['Tc'],
                             margin=self.cfg.max_fuel_temp_C-rad['Tc']))
            if rad['Tc'] <= self.cfg.max_fuel_temp_C: break
            # cannot converge by adjusting h alone; break and let outer loop fix geometry
            break
        self.results['heat'] = dict(
            rad=rad, pk_z=float(self.results['neutronics']['zc'][pk]),
            Tcenter=rad['Tc'],
            margin=self.cfg.max_fuel_temp_C-rad['Tc'],
            hist=pd.DataFrame(hist))

    def solve(self) -> Dict:
        self.neutron_physics()
        self._geometry()
        self.fluid_dynamics()
        self.heat_transfer()
        neu = self.results['neutronics']
        geo = self.results['geometry']
        fld = self.results['fluid']
        ht  = self.results['heat']
        self.results['summary'] = dict(
            core_D_m      = neu['D'],
            core_H_m      = neu['H'],
            core_V_m3     = neu['V'],
            n_assy        = geo['n_assy'],
            n_rods        = geo['n_rods'],
            rod_pitch_mm  = self.cfg.rod_pitch_m*1e3,
            fuel_d_mm     = self.cfg.fuel_pellet_diameter_m*1e3,
            AR            = self.cfg.aspect_ratio_H_over_D,
            m_dot_kg_s    = fld['m'],
            dp_kPa        = fld['dp']/1e3,
            Tout_C        = fld['T_bulk'][-1],
            T_center_C    = ht['Tcenter'],
            margin_C      = ht['margin'],
        )
        return self.results


# outer optimizer
def optimise_core(verbose=True) -> Tuple[PWRConfig, Dict, pd.DataFrame]:
    prop = WaterPropertyLibrary()
    outer_hist = []

    def objective(x):
        ar, pitch_mm, fuel_d_mm = x
        cfg = PWRConfig(
            aspect_ratio_H_over_D=ar,
            rod_pitch_m=pitch_mm*1e-3,
            fuel_pellet_diameter_m=fuel_d_mm*1e-3,
        )
        try:
            sol = PWRCoreSolver(cfg, prop)
            res = sol.solve()
            s   = res['summary']
            # penalised objective: volume + penalties for constraint violations
            temp_penalty = max(0, s['T_center_C']-cfg.max_fuel_temp_C)*1e-3
            dp_penalty   = max(0, s['dp_kPa']-cfg.delta_p_target_pa/1e3)*10
            geom_penalty = max(0, fuel_d_mm*1e-3*2 - pitch_mm*1e-3*0.85)*1e3
            obj = s['core_V_m3'] + temp_penalty + dp_penalty + geom_penalty
            outer_hist.append(dict(
                ar=ar, pitch_mm=pitch_mm, fuel_d_mm=fuel_d_mm,
                V_m3=s['core_V_m3'], dp_kPa=s['dp_kPa'],
                T_center_C=s['T_center_C'], margin_C=s['margin_C'],
                obj=obj))
            if verbose:
                print(f"  AR={ar:.3f} p={pitch_mm:.2f}mm d={fuel_d_mm:.2f}mm "
                      f"V={s['core_V_m3']:.3f}m³ T_c={s['T_center_C']:.1f}°C "
                      f"dp={s['dp_kPa']:.1f}kPa margin={s['margin_C']:.1f}°C")
            return obj
        except Exception as e:
            return 1e6

    # bounds: [AR, pitch(mm), fuel_d(mm)]
    bounds = [(0.70, 1.60), (11.0, 16.0), (4.5, 8.5)]
    lb = [b[0] for b in bounds]; ub = [b[1] for b in bounds]

    print("Global search (differential evolution)...")
    res_de = differential_evolution(objective, bounds,
                                    maxiter=30, popsize=8, tol=1e-4,
                                    mutation=(0.5,1.2), recombination=0.7,
                                    seed=42, workers=1, polish=False)
    x0 = res_de.x
    print(f"\nDE best → AR={x0[0]:.3f}, pitch={x0[1]:.2f}mm, d={x0[2]:.2f}mm, obj={res_de.fun:.4f}")

    print("\nLocal refinement (L-BFGS-B)...")
    res_loc = minimize(objective, x0, method='L-BFGS-B',
                       bounds=bounds, options=dict(ftol=1e-8, gtol=1e-6, maxiter=200))
    x_opt = res_loc.x
    print(f"Opt → AR={x_opt[0]:.3f}, pitch={x_opt[1]:.2f}mm, d={x_opt[2]:.2f}mm\n")

    best_cfg = PWRConfig(
        aspect_ratio_H_over_D=x_opt[0],
        rod_pitch_m=x_opt[1]*1e-3,
        fuel_pellet_diameter_m=x_opt[2]*1e-3,
    )
    final_sol = PWRCoreSolver(best_cfg, prop)
    final_res = final_sol.solve()
    hist_df   = pd.DataFrame(outer_hist)
    return best_cfg, final_res, hist_df, final_sol


# plots
def make_all_plots(best_cfg, final_res, hist_df, solver_obj, sweep_ars=None):
    Path('output').mkdir(exist_ok=True)

    # Outer-optimisation convergence
    fig_opt = make_subplots(rows=2, cols=2,
                            subplot_titles=['Core Volume (obj)',
                                            'Peak Fuel Temperature',
                                            'Pressure Drop',
                                            'Temperature Margin'])
    idx = list(range(1, len(hist_df)+1))
    fig_opt.add_trace(go.Scatter(x=idx, y=hist_df['V_m3'], mode='lines'), row=1, col=1)
    fig_opt.add_trace(go.Scatter(x=idx, y=hist_df['T_center_C'], mode='lines', name='T_c'), row=1, col=2)
    fig_opt.add_trace(go.Scatter(x=idx, y=[best_cfg.max_fuel_temp_C]*len(idx), mode='lines', name='limit', line=dict(dash='dash')), row=1, col=2)
    fig_opt.add_trace(go.Scatter(x=idx, y=hist_df['dp_kPa'], mode='lines', name='dp'), row=2, col=1)
    fig_opt.add_trace(go.Scatter(x=idx, y=[best_cfg.delta_p_target_pa/1e3]*len(idx), mode='lines', name='target', line=dict(dash='dash')), row=2, col=1)
    fig_opt.add_trace(go.Scatter(x=idx, y=hist_df['margin_C'], mode='lines', name='margin'), row=2, col=2)
    fig_opt.add_trace(go.Scatter(x=idx, y=[0]*len(idx), mode='lines', name='zero', line=dict(dash='dash')), row=2, col=2)
    fig_opt.update_layout(title='Outer optimisation convergence history', showlegend=False, height=700)
    fig_opt.write_html('output/optimisation_convergence.html', include_plotlyjs='cdn')

    # Fluid inner-loop: Re → Nu → h → T 
    fl_hist = final_res['fluid']['hist']
    fig_fl = make_subplots(rows=2, cols=2,
                           subplot_titles=['Reynolds (avg)', 'Nusselt (avg)',
                                           'h conv (avg, W/m²K)', 'Outlet Temperature'])
    it = fl_hist['iter']
    fig_fl.add_trace(go.Scatter(x=it, y=fl_hist['Re_avg'], mode='lines+markers'), row=1, col=1)
    fig_fl.add_trace(go.Scatter(x=it, y=fl_hist['Nu_avg'], mode='lines+markers'), row=1, col=2)
    fig_fl.add_trace(go.Scatter(x=it, y=fl_hist['h_avg'],  mode='lines+markers'), row=2, col=1)
    fig_fl.add_trace(go.Scatter(x=it, y=fl_hist['Tout'],   mode='lines+markers'), row=2, col=2)
    fig_fl.update_layout(title='Fluid iteration: Re → Nu → h → T  (inner loop)', showlegend=False, height=650)
    fig_fl.write_html('output/fluid_iteration_chain.html', include_plotlyjs='cdn')

    # Heat-transfer radial profile
    rad = final_res['heat']['rad']
    r_m = rad['r']*1e3          # mm
    T_f = rad['Tf']
    fig_ht = go.Figure()
    fig_ht.add_trace(go.Scatter(x=r_m, y=T_f, mode='lines', name='T fuel'))
    fig_ht.add_hline(y=best_cfg.max_fuel_temp_C, line_dash='dash', annotation_text='Max fuel T')
    fig_ht.add_hline(y=rad['Tc'], line_dash='dot', line_color='orange', annotation_text='Centerline')
    fig_ht.update_layout(title='Radial temperature profile at peak rod (sparse matrix solve)',
                         xaxis_title='Radius (mm)', yaxis_title='Temperature (°C)')
    fig_ht.write_html('output/radial_temp_profile.html', include_plotlyjs='cdn')

    # Axial coolant temperature
    zc  = final_res['neutronics']['zc']
    Tc  = final_res['fluid']['Tc']
    fig_ax = go.Figure()
    fig_ax.add_trace(go.Scatter(x=zc, y=Tc, mode='lines', name='T coolant'))
    fig_ax.add_trace(go.Scatter(x=zc, y=final_res['fluid']['q_lin']/1e3, mode='lines', name="q' (kW/m)", yaxis='y2'))
    fig_ax.update_layout(title="Axial coolant temperature and linear power",
                         xaxis_title='z (m)', yaxis_title='T (°C)',
                         yaxis2=dict(title="q' (kW/m)", overlaying='y', side='right'))
    fig_ax.write_html('output/axial_temp_power.html', include_plotlyjs='cdn')

    # Design sweep: AR vs T_center / volume
    if sweep_ars is None:
        sweep_ars = np.linspace(0.7, 1.6, 10)
    prop = WaterPropertyLibrary()
    sw_rows = []
    for ar in sweep_ars:
        cfg = PWRConfig(aspect_ratio_H_over_D=float(ar),
                        rod_pitch_m=best_cfg.rod_pitch_m,
                        fuel_pellet_diameter_m=best_cfg.fuel_pellet_diameter_m)
        try:
            s = PWRCoreSolver(cfg, prop); s.solve()
            sm = s.results['summary']
            sw_rows.append(dict(AR=ar, V=sm['core_V_m3'], Tc=sm['T_center_C'], dp=sm['dp_kPa']))
        except: pass
    sw = pd.DataFrame(sw_rows)
    fig_sw = make_subplots(rows=1, cols=2, subplot_titles=['Core volume vs AR','T center vs AR'])
    fig_sw.add_trace(go.Scatter(x=sw['AR'], y=sw['V'], mode='lines+markers', name='Vol'), row=1, col=1)
    fig_sw.add_trace(go.Scatter(x=sw['AR'], y=sw['Tc'], mode='lines+markers', name='Tc'), row=1, col=2)
    fig_sw.add_trace(go.Scatter(x=sw['AR'], y=[1000]*len(sw), mode='lines', name='limit', line=dict(dash='dash')), row=1, col=2)
    fig_sw.add_vline(x=best_cfg.aspect_ratio_H_over_D, line_dash='dot', annotation_text='opt', row=1, col=1)
    fig_sw.add_vline(x=best_cfg.aspect_ratio_H_over_D, line_dash='dot', row=1, col=2)
    fig_sw.update_layout(title='Design sweep at optimised pitch & fuel diameter', showlegend=False, height=450)
    fig_sw.write_html('output/design_sweep_plots.html', include_plotlyjs='cdn')

    # Animated 3D flux surface as geometry expands
    anim_ars = np.round(np.linspace(0.70, 1.60, 12), 2)
    frames = []; first = None
    for ar in anim_ars:
        cfg = PWRConfig(aspect_ratio_H_over_D=float(ar), nr=40, nz=60,
                        rod_pitch_m=best_cfg.rod_pitch_m,
                        fuel_pellet_diameter_m=best_cfg.fuel_pellet_diameter_m)
        sol = PWRCoreSolver(cfg, prop); sol.neutron_physics()
        neu = sol.results['neutronics']
        r, z, fs = neu['rc'], neu['zc'], neu['fs']
        R, H = neu['R'], neu['H']
        rf = np.linspace(0, R, 60); zf = np.linspace(-H/2, H/2, 80)
        interp = RegularGridInterpolator((r, z), fs, bounds_error=False, fill_value=0)
        Rf, Zf = np.meshgrid(rf, zf, indexing='ij')
        phi = interp(np.column_stack([Rf.ravel(), Zf.ravel()])).reshape(Rf.shape)
        surf = go.Surface(x=Rf, y=Zf, z=phi, colorscale='Plasma', cmin=0, cmax=1, showscale=True,
                          colorbar=dict(title='Norm flux'))
        if first is None: first = [surf]
        frames.append(go.Frame(data=[surf], name=f'AR {ar:.2f}',
                               layout=go.Layout(title_text=f'AR={ar:.2f}  D={2*R:.3f}m  H={H:.3f}m')))
    steps = [dict(method='animate', label=f.name,
                  args=[[f.name], {'frame': {'duration': 500, 'redraw': True}, 'mode': 'immediate'}])
             for f in frames]
    fig_anim = go.Figure(data=first, frames=frames)
    fig_anim.update_layout(
        title='3D Neutron Flux Surface — Animated Geometry Sweep',
        scene=dict(xaxis_title='r (m)', yaxis_title='z (m)', zaxis_title='Norm flux',
                   camera=dict(eye=dict(x=1.6, y=-1.4, z=0.9))),
        updatemenus=[dict(type='buttons', y=-0.08, x=0.5, xanchor='center',
                          buttons=[
                              dict(label='▶ Play', method='animate',
                                   args=[None, {'frame': {'duration': 600, 'redraw': True}, 'fromcurrent': True}]),
                              dict(label='⏸ Pause', method='animate',
                                   args=[[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}])])],
        sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix='AR: ', visible=True, xanchor='center'),
                      x=0.05, len=0.9, y=-0.02, pad=dict(t=40))],
        width=1000, height=700)
    fig_anim.write_html('output/flux_animation_3d.html', include_plotlyjs='cdn')

def plot_core_xy(core, flux_contour=True, show_rods=True,
                 rod_stride=1, savepath=None):
    """
    Top-down (x-y) axial cross-section of core.

    core:         Core instance
    flux_contour: overlay J0(2.405 r / R_core) contours
    show_rods:    draw individual rod circles (slow for 7k+ rods)
    rod_stride:   draw every Nth rod if show_rods (1 = all)
    savepath:     if given, save to path
    """
    fig, ax = plt.subplots(figsize=(9, 9))

    # --- flux contour backdrop ---
    if flux_contour:
        R = core.R_core
        N = 300
        xs = np.linspace(-R, R, N)
        ys = np.linspace(-R, R, N)
        X, Y = np.meshgrid(xs, ys)
        r = np.hypot(X, Y)
        from scipy.special import j0
        phi = np.where(r <= R, j0(2.405 * r / R), np.nan)
        cf = ax.contourf(X, Y, phi, levels=15, cmap="viridis", alpha=0.55)
        plt.colorbar(cf, ax=ax, label=r"$\phi / \phi_{max}$  (radial, J$_0$)")

    # --- assembly bounding boxes ---
    for a in core.assemblies:
        xmin, ymin, xmax, ymax = a.bbox
        ax.add_patch(Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            fill=False, edgecolor="white", linewidth=1.4, zorder=2,
        ))
        ax.plot(a.cx, a.cy, "w+", markersize=6, zorder=3)

    # --- rods ---
    if show_rods:
        r_rod = core.D_rod / 2
        for k, p in enumerate(core.all_pins):
            if k % rod_stride != 0:
                continue
            ax.add_patch(Circle(
                (p.x, p.y), r_rod,
                facecolor="black", edgecolor="none", zorder=4,
            ))

    # --- core circumscribed circle (flux outer radius R) ---
    ax.add_patch(Circle(
        (0, 0), core.R_core,
        fill=False, edgecolor="red", linewidth=2.2,
        linestyle="--", zorder=5, label=f"R_core = {core.R_core*1e3:.1f} mm",
    ))

    # --- cosmetics ---
    pad = 0.08 * core.R_core
    ax.set_xlim(-core.R_core - pad, core.R_core + pad)
    ax.set_ylim(-core.R_core - pad, core.R_core + pad)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(
        f"Core cross-section  |  {core.N_assemblies} assemblies  "
        f"|  {core.n_pins_total} rods  |  D_core = {2*core.R_core:.3f} m"
    )
    ax.legend(loc="upper right")
    ax.grid(alpha=0.25)

    if savepath:
        fig.savefig(savepath, dpi=150, bbox_inches="tight")
    return fig, ax


    print("All plots written to output/")


# main
if __name__ == '__main__':
    Path('output').mkdir(exist_ok=True)
    best_cfg, final_res, hist_df, solver_obj = optimise_core(verbose=True)
    hist_df.to_csv('output/optimisation_history.csv', index=False)
    final_res['fluid']['hist'].to_csv('output/fluid_iteration_chain.csv', index=False)
    final_res['heat']['hist'].to_csv('output/heat_iteration_chain.csv', index=False)
    pd.DataFrame([final_res['summary']]).to_csv('output/optimal_design_summary.csv', index=False)
    print("\n=== OPTIMAL DESIGN ===")

Global search (differential evolution)...
  AR=0.894 p=13.12mm d=5.73mm V=2.426m³ T_c=1187.1°C dp=175.0kPa margin=-187.1°C
  AR=1.226 p=11.24mm d=8.37mm V=2.426m³ T_c=1200.3°C dp=175.0kPa margin=-200.3°C
  AR=1.367 p=11.63mm d=4.62mm V=2.426m³ T_c=1393.4°C dp=175.0kPa margin=-393.4°C
  AR=1.148 p=14.97mm d=5.87mm V=2.426m³ T_c=1966.2°C dp=175.0kPa margin=-966.2°C
  AR=1.465 p=15.22mm d=7.48mm V=2.426m³ T_c=2716.3°C dp=175.0kPa margin=-1716.3°C
  AR=0.760 p=13.52mm d=6.75mm V=2.426m³ T_c=1062.4°C dp=175.0kPa margin=-62.4°C
  AR=0.839 p=12.11mm d=4.69mm V=2.426m³ T_c=941.2°C dp=175.0kPa margin=58.8°C
  AR=1.420 p=12.83mm d=6.47mm V=2.426m³ T_c=1834.2°C dp=175.0kPa margin=-834.2°C
  AR=1.287 p=12.31mm d=5.42mm V=2.426m³ T_c=1475.1°C dp=175.0kPa margin=-475.1°C
  AR=1.161 p=13.00mm d=4.93mm V=2.426m³ T_c=1525.0°C dp=175.0kPa margin=-525.0°C
  AR=1.017 p=14.19mm d=7.72mm V=2.426m³ T_c=1599.4°C dp=175.0kPa margin=-599.4°C
  AR=1.204 p=14.74mm d=8.09mm V=2.426m³ T_c=2036.1°C dp=175.0kPa margi

ModuleNotFoundError: No module named 'config'